## GOOGLE COLAB SETUP

IMPORTANT: Run this cell first in Colab to mount Drive and navigate to correct directory.

This notebook performs Monte Carlo Dropout uncertainty quantification on trained GNN models.

Expected Runtime: ~20-25 minutes (with GPU)

Outputs:
- Overall model validation metrics
- Single scenario uncertainty maps
- Network-wide average uncertainty heatmap

In [ ]:
# GOOGLE COLAB SETUP - Mount Drive and Navigate
from google.colab import drive
drive.mount('/content/drive')

# Navigate to notebook directory
%cd /content/drive/MyDrive/Zamin-thesis/ml_surrogates_for_agent_based_transport_models-main/scripts/misc

print("Drive mounted successfully")
print(f"Current directory: {os.getcwd()}")

<!-- ABSTRACT -->

With this script, we apply monte carlo dropout to the trained model and check how well it performs. The result is a plot of the uncertainty of the model's predictions. However, it seems that the uncertainty is not very high.

In [ ]:
import os
import sys
import json
import joblib

import numpy as np
from tqdm import tqdm
import geopandas as gpd

import torch

# Add the 'scripts' directory to Python Path
scripts_path=os.path.abspath(os.path.join(os.getcwd(), '..'))
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

import evaluation.help_functions as hf
import evaluation.plot_functions as pf

from gnn.help_functions import mc_dropout_predict
from gnn.models.point_net_transf_gat import PointNetTransfGAT
from data_preprocessing.help_functions import highway_mapping

In [ ]:
# Get the absolute path to the project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

# Paths - UPDATED FOR TRIAL 8 (BEST MODEL)
run_path = os.path.join(project_root, "data", "runs_01_2025", "run_trial_8_best_model")  # Trial 8: R²=0.5957, Gap=0.2%
districts = gpd.read_file(os.path.join(project_root, "data", "visualisation", "districts_paris.geojson"))
base_case_path = os.path.join(project_root, "data", "links_and_stats", "pop_1pct_basecase_average_output_links.geojson")
result_path = 'results/'

# GNN Parameters (Trial 8 configuration)
point_net_conv_layer_structure_local_mlp="256"
point_net_conv_layer_structure_global_mlp = "512"
gat_conv_layer_structure = "128,256,512"
dropout = 0.2  # Trial 8 used dropout=0.2
use_dropout = False  # MC dropout will enable this during inference
predict_mode_stats = False
in_channels = 5
out_channels = 1

links_base_case = gpd.read_file(base_case_path, crs="EPSG:4326")
data_created_during_training = os.path.join(run_path, 'data_created_during_training')

print("="*60)
print("CONFIGURATION LOADED")
print("="*60)
print(f"Trial: Trial 8 (Best Model)")
print(f"Expected R²: 0.5957")
print(f"Expected Gap: 0.2%")
print(f"Dropout: {dropout}")
print("="*60)

In [ ]:
###########################################
### Load test data from the run itself! ###
###########################################

# Load scalers
scaler_x = joblib.load(os.path.join(data_created_during_training, 'test_x_scaler.pkl'))
scaler_pos = joblib.load(os.path.join(data_created_during_training, 'test_pos_scaler.pkl'))

# Load the test dataset created during training
test_set_dl = torch.load(os.path.join(data_created_during_training, 'test_dl.pt'))

# Load the DataLoader parameters
with open(os.path.join(data_created_during_training, 'test_loader_params.json'), 'r') as f:
    test_set_dl_loader_params = json.load(f)
    
# Remove or correct collate_fn if it is incorrectly specified
if 'collate_fn' in test_set_dl_loader_params and isinstance(test_set_dl_loader_params['collate_fn'], str):
    del test_set_dl_loader_params['collate_fn']  # Remove it to use the default collate function
    
test_set_loader = torch.utils.data.DataLoader(test_set_dl, **test_set_dl_loader_params)

In [ ]:
# VERIFY PATHS EXIST
print("\n" + "="*60)
print("PATH VERIFICATION")
print("="*60)
print(f"Project Root: {project_root}")
print(f"  Exists: {os.path.exists(project_root)}")
print(f"\nRun Path: {run_path}")
print(f"  Exists: {os.path.exists(run_path)}")
print(f"\nModel File: {os.path.join(run_path, 'trained_model/model.pth')}")
print(f"  Exists: {os.path.exists(os.path.join(run_path, 'trained_model/model.pth'))}")
print(f"\nTest Data: {data_created_during_training}")
print(f"  Exists: {os.path.exists(data_created_during_training)}")
print(f"\nBase Case GeoJSON:")
print(f"  Exists: {os.path.exists(base_case_path)}")
print("="*60)

if not os.path.exists(run_path):
    print("\nWARNING: Run path does not exist!")
    print("Available trials in runs_01_2025:")
    runs_dir = os.path.join(project_root, "data", "runs_01_2025")
    if os.path.exists(runs_dir):
        print([d for d in os.listdir(runs_dir) if os.path.isdir(os.path.join(runs_dir, d))])
    else:
        print("ERROR: runs_01_2025 directory not found!")
else:
    print("\nAll paths verified successfully!")

In [ ]:
point_net_conv_layer_structure_local_mlp = [int(x) for x in point_net_conv_layer_structure_local_mlp.split(',')]
point_net_conv_layer_structure_global_mlp = [int(x) for x in point_net_conv_layer_structure_global_mlp.split(',')]
gat_conv_layer_structure = [int(x) for x in gat_conv_layer_structure.split(',')]

model = PointNetTransfGAT(in_channels=in_channels, out_channels=out_channels,
              point_net_conv_layer_structure_local_mlp=point_net_conv_layer_structure_local_mlp, 
              point_net_conv_layer_structure_global_mlp = point_net_conv_layer_structure_global_mlp,
              gat_conv_layer_structure=gat_conv_layer_structure,
              dropout=dropout,
              use_dropout=use_dropout,
              predict_mode_stats=predict_mode_stats)

# Load the model state dictionary
model_path = os.path.join(run_path, 'trained_model/model.pth')
model.load_state_dict(torch.load(model_path), strict=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

loss_fct = torch.nn.MSELoss().to(dtype=torch.float32).to(device)

In [ ]:
test_loss, r_squared, actual_vals, predictions, baseline_loss = hf.validate_model_on_test_set(model, test_set_loader.dataset, loss_fct, device)

print(f"Test Loss: {test_loss}")
print(f"R-squared: {r_squared}")
print(f"Baseline Loss: {baseline_loss}")

### Next, we will look at single elements of the test set and visualize the performance of the model.


In [ ]:
i = 2 # index from the test set, pick a particular sample

fixed_norm_max = 50
    
my_test_data = test_set_loader.dataset[i]
my_test_x = test_set_loader.dataset[i].x
my_test_x = my_test_x.to('cpu')

test_loss_my_test_data, r_squared_my_test_data, actual_vals_my_test_data, predictions_my_test_data, baseline_loss_my_test_data = hf.validate_model_on_test_set(model, my_test_data, loss_fct, device)
print(f"Sample {i}")
print(f"Test Loss: {test_loss_my_test_data}")
print(f"R-squared: {r_squared_my_test_data}")
print(f"Baseline Loss: {baseline_loss_my_test_data}")

inversed_x = scaler_x.inverse_transform(my_test_x)

gdf_with_og_values = hf.data_to_geodataframe_with_og_values(data=my_test_data, original_gdf=links_base_case, predicted_values=predictions_my_test_data, inversed_x=inversed_x)
gdf_with_og_values['capacity_reduction_rounded'] = gdf_with_og_values['capacity_reduction'].round(decimals=3)
gdf_with_og_values['highway'] = gdf_with_og_values['highway'].map(highway_mapping)

# gdf_with_og_values['district'] = gdf_with_og_values.apply(lambda row: districts[districts.contains(row.geometry)].iloc[0]['c_ar'] if not districts[districts.contains(row.geometry)].empty else 'Unknown', axis=1)
# gdf_with_og_values = gpd.sjoin(gdf_with_og_values, districts, how='left', op='intersects')

print(f"\nPredicted:")
pf.plot_combined_output(gdf_input=gdf_with_og_values, column_to_plot="vol_car_change_predicted", 
                        save_it=False, number_to_plot=i, zone_to_plot="this zone", is_predicted=True, alpha=0, use_fixed_norm=True, 
                        fixed_norm_max=fixed_norm_max, known_districts=False, buffer=0.0005, districts_of_interest=None,
                        plot_contour_lines=True, plot_policy_roads=False, result_path=result_path, with_legend=False)

print(f"Actual:")
pf.plot_combined_output(gdf_input=gdf_with_og_values, column_to_plot="vol_car_change_actual", save_it=False, 
                        number_to_plot=i, zone_to_plot="this zone", is_predicted=False,alpha=10,use_fixed_norm=True, 
                        fixed_norm_max=fixed_norm_max, known_districts=False, buffer=0.0005, districts_of_interest=None,
                        plot_contour_lines=True, plot_policy_roads=False, result_path=result_path, with_legend=False)

In [ ]:
# MC DROPOUT on Single Sample

i = 32
test_data = test_set_loader.dataset[i]
test_x = test_set_loader.dataset[i].x
test_x = test_x.to('cpu')

test_loss, r_squared, actual_vals, predictions, baseline_loss = hf.validate_model_on_test_set(model, test_data, loss_fct, device)
print(f"Test {i}")
print(f"Test Loss: {test_loss}")
print(f"R-squared: {r_squared}")
print(f"Baseline Loss: {baseline_loss}")

inversed_x = scaler_x.inverse_transform(test_x)
mean_predictions, uncertainties = mc_dropout_predict(model, test_data, num_samples=50, device=device)

gdf_with_og_values = hf.data_to_geodataframe_with_og_values(data=test_data, original_gdf=links_base_case, predicted_values=predictions, inversed_x=inversed_x, use_all_features=False)
gdf_with_og_values['capacity_reduction_rounded'] = gdf_with_og_values['capacity_reduction'].round(decimals=3)
gdf_with_og_values['highway'] = gdf_with_og_values['highway'].map(highway_mapping)
gdf_with_og_values['mc_uncertainty'] = uncertainties

pf.plot_combined_output(gdf_input=gdf_with_og_values, column_to_plot="mc_uncertainty", plot_contour_lines=False,
                        save_it=False, number_to_plot=i, zone_to_plot="this zone", is_predicted=True, use_fixed_norm=False,
                        known_districts=False, buffer=0.0005, districts_of_interest=None, cmap='Reds')

In [ ]:
# MC DROPOUT on entire test set

mean_uncertainties = []

for i in tqdm(range(len(test_set_loader.dataset))):
    
    test_data = test_set_loader.dataset[i]
    test_x = test_set_loader.dataset[i].x
    test_x = test_x.to('cpu')

    mean_predictions, uncertainties = mc_dropout_predict(model, test_data, num_samples=50, device=device)
    mean_uncertainties.append(uncertainties)

mean_uncertainties = np.array(mean_uncertainties).mean(axis=0)

In [ ]:
# On the last sample, but does not matter
inversed_x = scaler_x.inverse_transform(test_x)
gdf_with_og_values = hf.data_to_geodataframe_with_og_values(data=test_data, original_gdf=links_base_case, predicted_values=mean_predictions, inversed_x=inversed_x, use_all_features=False)
gdf_with_og_values['capacity_reduction_rounded'] = gdf_with_og_values['capacity_reduction'].round(decimals=3)
gdf_with_og_values['highway'] = gdf_with_og_values['highway'].map(highway_mapping)
gdf_with_og_values['mc_uncertainty'] = mean_uncertainties

pf.plot_combined_output(gdf_input=gdf_with_og_values, column_to_plot="mc_uncertainty", plot_contour_lines=False,
                        save_it=False, number_to_plot=i+1, zone_to_plot="this zone", is_predicted=True, use_fixed_norm=False,
                        known_districts=False, buffer=0.0005, districts_of_interest=None, cmap='Reds')

## QUANTITATIVE UNCERTAINTY ANALYSIS

Calculate detailed statistics and breakdowns of uncertainty patterns.

In [ ]:
# Overall Uncertainty Statistics
print("="*80)
print(" UNCERTAINTY STATISTICS (Network-Wide Average)")
print("="*80)
print(f"Mean Uncertainty:     {mean_uncertainties.mean():.6f}")
print(f"Std Uncertainty:      {mean_uncertainties.std():.6f}")
print(f"Max Uncertainty:      {mean_uncertainties.max():.6f}")
print(f"Min Uncertainty:      {mean_uncertainties.min():.6f}")
print(f"Median Uncertainty:   {np.median(mean_uncertainties):.6f}")
print(f"\nPercentiles:")
print(f"  95th Percentile:    {np.percentile(mean_uncertainties, 95):.6f}")
print(f"  75th Percentile:    {np.percentile(mean_uncertainties, 75):.6f}")
print(f"  50th Percentile:    {np.percentile(mean_uncertainties, 50):.6f}")
print(f"  25th Percentile:    {np.percentile(mean_uncertainties, 25):.6f}")
print(f"   5th Percentile:    {np.percentile(mean_uncertainties, 5):.6f}")
print("="*80)

In [ ]:
# Uncertainty by Road Type
import matplotlib.pyplot as plt

uncertainty_by_highway = gdf_with_og_values.groupby('highway')['mc_uncertainty'].agg(['mean', 'std', 'count'])
uncertainty_by_highway = uncertainty_by_highway.sort_values('mean', ascending=False)

print("\n" + "="*80)
print(" UNCERTAINTY BY ROAD TYPE")
print("="*80)
print(uncertainty_by_highway.to_string())
print("="*80)

# Plot horizontal bar chart
fig, ax = plt.subplots(figsize=(12, 6))
uncertainty_by_highway['mean'].plot(kind='barh', ax=ax, color='crimson', alpha=0.7, edgecolor='darkred', linewidth=1.5)
ax.set_xlabel('Mean Uncertainty', fontsize=12, fontweight='bold')
ax.set_ylabel('Road Type', fontsize=12, fontweight='bold')
ax.set_title('Average MC Dropout Uncertainty by Road Type\n(Trial 8 - Best Model)', fontsize=14, fontweight='bold', pad=15)
ax.grid(axis='x', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

print(f"\nKey Finding: {uncertainty_by_highway.index[0]} roads show highest uncertainty ({uncertainty_by_highway.iloc[0]['mean']:.6f})")
print(f"Key Finding: {uncertainty_by_highway.index[-1]} roads show lowest uncertainty ({uncertainty_by_highway.iloc[-1]['mean']:.6f})")

In [ ]:
# High Uncertainty Links Analysis
threshold_90 = np.percentile(mean_uncertainties, 90)  # Top 10%
threshold_95 = np.percentile(mean_uncertainties, 95)  # Top 5%

high_uncertainty_90 = gdf_with_og_values[gdf_with_og_values['mc_uncertainty'] > threshold_90]
high_uncertainty_95 = gdf_with_og_values[gdf_with_og_values['mc_uncertainty'] > threshold_95]

print("\n" + "="*80)
print(" HIGH UNCERTAINTY LINKS IDENTIFICATION")
print("="*80)
print(f"Total Network Links: {len(gdf_with_og_values)}")
print(f"\nTop 10% Uncertainty Threshold: {threshold_90:.6f}")
print(f"  Links above threshold: {len(high_uncertainty_90)} ({len(high_uncertainty_90)/len(gdf_with_og_values)*100:.1f}%)")
print(f"\nTop 5% Uncertainty Threshold: {threshold_95:.6f}")
print(f"  Links above threshold: {len(high_uncertainty_95)} ({len(high_uncertainty_95)/len(gdf_with_og_values)*100:.1f}%)")

print(f"\nHigh Uncertainty Links by Road Type (Top 10%):")
highway_breakdown = high_uncertainty_90.groupby('highway').size().sort_values(ascending=False)
for road_type, count in highway_breakdown.items():
    pct = count / len(high_uncertainty_90) * 100
    print(f"  {road_type:20s}: {count:4d} links ({pct:5.1f}%)")

print("="*80)
print(f"\nDEPLOYMENT RECOMMENDATION:")
print(f"   Flag {len(high_uncertainty_90)} links ({len(high_uncertainty_90)/len(gdf_with_og_values)*100:.1f}%) for manual review")
print(f"   These links require human oversight or additional data collection")
print("="*80)

In [ ]:
# Uncertainty Distribution Histogram
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
axes[0].hist(mean_uncertainties, bins=50, color='crimson', alpha=0.7, edgecolor='darkred', linewidth=1.2)
axes[0].axvline(mean_uncertainties.mean(), color='blue', linestyle='--', linewidth=2.5, label=f'Mean: {mean_uncertainties.mean():.6f}')
axes[0].axvline(np.median(mean_uncertainties), color='green', linestyle='--', linewidth=2.5, label=f'Median: {np.median(mean_uncertainties):.6f}')
axes[0].axvline(threshold_90, color='orange', linestyle='--', linewidth=2, label=f'90th Percentile: {threshold_90:.6f}')
axes[0].set_xlabel('Uncertainty Value', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Frequency (Number of Links)', fontsize=12, fontweight='bold')
axes[0].set_title('Distribution of MC Dropout Uncertainty\n(Network-Wide Average)', fontsize=13, fontweight='bold', pad=12)
axes[0].legend(fontsize=10, loc='upper right')
axes[0].grid(alpha=0.3, linestyle='--')

# Box plot by road type (top 8 road types)
top_highways = uncertainty_by_highway.head(8).index
data_for_boxplot = [gdf_with_og_values[gdf_with_og_values['highway'] == hw]['mc_uncertainty'].values for hw in top_highways]
bp = axes[1].boxplot(data_for_boxplot, labels=top_highways, patch_artist=True, vert=False)
for patch in bp['boxes']:
    patch.set_facecolor('crimson')
    patch.set_alpha(0.6)
axes[1].set_xlabel('Uncertainty Value', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Road Type', fontsize=12, fontweight='bold')
axes[1].set_title('Uncertainty Distribution by Road Type\n(Top 8 Road Types)', fontsize=13, fontweight='bold', pad=12)
axes[1].grid(axis='x', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print("Uncertainty distribution visualizations generated successfully")